# 実験: Spot Spread モデル (Exp-SpotSpread)

会合付きOISとスポットテナーOIS（1年物 T12）の差分 `S{n} = M{n} - T12` の変化を予測するモデルを検証します。

## 目的
- 現行 RV Butterfly モデルと比較して、Spot Spread モデルの予測性能（CS IC）を確認する。
- T12（連続テナー曲線）に対する割高/割安を予測することで、アルファの源泉を多角化できるか検討する。

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.processing import load_and_clean_data
from src.features_rv import generate_rv_features
from src.pooling_butterfly import pool_butterfly_data
from src.pooling_spot_spread import pool_spot_spread_data
from src.modeling import walk_forward_validation, summarize_ic, walk_forward_with_model

START_DATE = '2024-01-01'
print(f'Start Date: {START_DATE}')

Start Date: 2024-01-01


In [2]:
# データ読み込みと共通特徴量生成
df_raw = load_and_clean_data('../data/BOJ_data.xlsx', '../data/BOJ_meeting_history.csv')
df_feat = generate_rv_features(df_raw)
print(f'Data loaded: {len(df_raw)} rows')

Data loaded: 4653 rows


## 1. RV Butterfly モデルの再現（ベースライン）

In [3]:
print('--- Reproducing RV Butterfly ---')
df_fly = pool_butterfly_data(df_feat)

res_fly_3d = walk_forward_validation(df_fly, 'Target_3d_norm', START_DATE)
res_fly_5d = walk_forward_validation(df_fly, 'Target_5d_norm', START_DATE)

ic_fly_3d = summarize_ic(res_fly_3d, instrument_indices=set(range(2, 8)))
ic_fly_5d = summarize_ic(res_fly_5d, instrument_indices=set(range(2, 8)))

print(f'[Butterfly 3d] CS IC: {ic_fly_3d["cs_ic"]: .4f}')
print(f'[Butterfly 5d] CS IC: {ic_fly_5d["cs_ic"]: .4f}')

--- Reproducing RV Butterfly ---


[Butterfly 3d] CS IC:  0.3045
[Butterfly 5d] CS IC:  0.3459


## 2. Spot Spread モデルの検証

In [4]:
print('--- Running Spot Spread Model ---')
df_spot = pool_spot_spread_data(df_feat)

res_spot_3d = walk_forward_validation(df_spot, 'Target_3d_norm', START_DATE)
res_spot_5d = walk_forward_validation(df_spot, 'Target_5d_norm', START_DATE)

ic_spot_3d = summarize_ic(res_spot_3d, instrument_indices=set(range(1, 9)))
ic_spot_5d = summarize_ic(res_spot_5d, instrument_indices=set(range(1, 9)))

print(f'[Spot Spread 3d] CS IC: {ic_spot_3d["cs_ic"]: .4f}')
print(f'[Spot Spread 5d] CS IC: {ic_spot_5d["cs_ic"]: .4f}')

--- Running Spot Spread Model ---


[Spot Spread 3d] CS IC:  0.0414
[Spot Spread 5d] CS IC:  0.0909


## 3. 特徴量重要度の確認

In [5]:
_, model_spot_5d, X_test, _, _ = walk_forward_with_model(df_spot, 'Target_5d_norm', START_DATE)

importances = pd.DataFrame({
    'feature': model_spot_5d.feature_name(),
    'importance': model_spot_5d.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print('\nTop 10 Features (Spot Spread 5d):')
print(importances.head(10))


Top 10 Features (Spot Spread 5d):
                feature    importance
7          S4_frac_diff  15284.314100
1           Days_to_MPM  12431.136698
12           T12_spread  11878.238816
3          Spread_Level  10566.146257
8          S5_frac_diff   9582.822440
5          S2_frac_diff   8950.967013
20  Nikkei225_frac_diff   5658.773726
6          S3_frac_diff   4290.915585
10         S7_frac_diff   4012.381091
18     USDJPY_frac_diff   3972.235197


## 4. 比較まとめ

In [6]:
summary = pd.DataFrame({
    'Model': ['RV Butterfly', 'Spot Spread'],
    'CS IC 3d': [ic_fly_3d['cs_ic'], ic_spot_3d['cs_ic']],
    'CS IC 5d': [ic_fly_5d['cs_ic'], ic_spot_5d['cs_ic']],
    'Global IC 3d': [ic_fly_3d['ic_all'], ic_spot_3d['ic_all']],
    'Global IC 5d': [ic_fly_5d['ic_all'], ic_spot_5d['ic_all']]
})
print(summary)

          Model  CS IC 3d  CS IC 5d  Global IC 3d  Global IC 5d
0  RV Butterfly    0.3045    0.3459        0.3732        0.4185
1   Spot Spread    0.0414    0.0909        0.2475        0.3964
